# Filtering, Convolution, and Noise

> **Beginner · Image processing**


## Why this matters

Filtering is the foundation of denoising, sharpening, edge detection, and many learned vision operations. Kernel intuition matters more than memorizing functions.

**Where it appears:** Camera-noise cleanup, document preprocessing, smoothing before segmentation, and visual effects.


## Learning Objectives

- Understand convolution/correlation as the basis of all linear filters
- Implement and compare box, Gaussian, median, and bilateral filtering
- Choose the right filter for a given noise type


## Prerequisites

06 Drawing and Geometric Transformations

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.filter2D`, `cv2.blur`, `cv2.GaussianBlur`, `cv2.medianBlur`, `cv2.bilateralFilter`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Image Filtering and Convolution

Filtering slides a kernel over the image and combines neighboring pixel
values. **Linear** filters (box, Gaussian) blur everything uniformly.
**Median** filtering is non-linear and excellent at removing salt-and-pepper
noise while preserving edges. **Bilateral** filtering is edge-preserving:
it blurs based on both spatial AND intensity distance, smoothing flat
regions while keeping edges sharp -- at a real computational cost.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Manual convolution to build intuition

Before relying on `cv2.filter2D`, implement a small manual convolution to make explicit what the kernel actually does to each neighborhood.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid, Timer


def manual_convolve_center_pixel(
    image_gray: np.ndarray, kernel: np.ndarray, y: int, x: int
) -> float:
    """Compute a single output pixel by hand, for teaching purposes only (not efficient)."""
    kh, kw = kernel.shape
    half_h, half_w = kh // 2, kw // 2
    patch = image_gray[y - half_h : y + half_h + 1, x - half_w : x + half_w + 1].astype(
        np.float32
    )
    return float((patch * kernel).sum())


gray = cv2.cvtColor(
    load_real_image("images/standard", "peppers.jpg"), cv2.COLOR_BGR2GRAY
)
sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
manual_value = manual_convolve_center_pixel(gray, sharpen_kernel, 100, 100)

# Confirm it matches cv2.filter2D at the same location
full_result = cv2.filter2D(gray, -1, sharpen_kernel)
print(f"Manual: {manual_value:.1f}  cv2.filter2D: {int(full_result[100, 100])}")

### 2. Comparing blur filters on Gaussian noise

Box and Gaussian blur both reduce Gaussian-distributed noise well; Gaussian blur weights nearby pixels more, giving a more natural result for the same kernel size.


In [ ]:
def compare_blurs(image: np.ndarray, ksize: int = 7) -> dict:
    return {
        "box": cv2.blur(image, (ksize, ksize)),
        "gaussian": cv2.GaussianBlur(image, (ksize, ksize), 0),
    }


img = load_real_image("images/standard", "peppers.jpg")
noise = np.random.normal(0, 25, img.shape)
noisy_gaussian = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
blurred = compare_blurs(noisy_gaussian)
show_grid(
    [
        ("gaussian noise", noisy_gaussian),
        ("box blur", blurred["box"]),
        ("gaussian blur", blurred["gaussian"]),
    ]
)

### 3. Median filtering for salt-and-pepper noise

Median filtering is dramatically better than linear blur for impulse (salt-and-pepper) noise, because a single extreme outlier can't drag a median the way it drags a mean.


In [ ]:
def salt_and_pepper(image: np.ndarray, amount: float = 0.05) -> np.ndarray:
    out = image.copy()
    h, w = image.shape[:2]
    n = int(amount * h * w)
    ys = np.random.randint(0, h, n)
    xs = np.random.randint(0, w, n)
    out[ys[: n // 2], xs[: n // 2]] = 255
    out[ys[n // 2 :], xs[n // 2 :]] = 0
    return out


sp_noisy = salt_and_pepper(load_real_image("images/standard", "peppers.jpg"))
gaussian_on_sp = cv2.GaussianBlur(sp_noisy, (5, 5), 0)
median_on_sp = cv2.medianBlur(sp_noisy, 5)

show_grid(
    [
        ("salt & pepper noise", sp_noisy),
        ("gaussian blur (poor fit)", gaussian_on_sp),
        ("median blur (correct fit)", median_on_sp),
    ]
)

### 4. Bilateral filtering: edge-preserving smoothing

`cv2.bilateralFilter` smooths flat regions while keeping edges sharp, at higher computational cost -- benchmark it against Gaussian blur to see the trade-off.


In [ ]:
with Timer("Gaussian blur"):
    g = cv2.GaussianBlur(noisy_gaussian, (9, 9), 0)

with Timer("Bilateral filter"):
    b = cv2.bilateralFilter(noisy_gaussian, d=9, sigmaColor=75, sigmaSpace=75)

show_grid([("gaussian (blurs edges too)", g), ("bilateral (edges preserved)", b)])

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Image Filtering and Convolution: Artistic Pencil Sketch Filter

By combining edge filters with bilateral smoothing and blending operations, we can create an artistic pencil-sketch effect. This process highlights edges while smoothing interior textures to simulate handmade pencil strokes.


In [ ]:
# Convert frame to grayscale
img = load_real_image("images/standard", "peppers.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Step 1: Smooth image using bilateral filtering to remove noise while keeping edges
smoothed = cv2.bilateralFilter(gray, 9, 75, 75)

# Step 2: Extract sharp boundaries using Laplacian operator
edges = cv2.Laplacian(smoothed, cv2.CV_8U, ksize=5)

# Step 3: Invert edge mask to make lines dark on a light background
sketch = cv2.bitwise_not(edges)

print("Pencil sketch filter generated successfully.")
show_grid(
    [("Original BGR", img), ("Smoothed Grayscale", smoothed), ("Sketch Effect", sketch)]
)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Image Filtering and Convolution
1. Implement a manual convolution over a small full patch (not just one pixel) and compare timing to `cv2.filter2D`.
2. Sweep bilateral filter's `sigmaColor` from 10 to 150 and observe how edge preservation changes.
3. Combine median filtering (for salt-and-pepper) followed by bilateral filtering (for remaining smooth noise) into one pipeline function.

Use the empty cell below to work through them.


#### Solutions — Image Filtering and Convolution

In [ ]:
# Solution 1: Manual convolution vs cv2.filter2D comparison
def benchmark_convolution() -> None:
    img = np.random.randint(0, 256, (100, 100), dtype=np.uint8)
    kernel = np.ones((3, 3), dtype=np.float32) / 9.0

    # 1. cv2.filter2D execution
    t0 = time.perf_counter()
    res_cv = cv2.filter2D(img, -1, kernel)
    t_cv = (time.perf_counter() - t0) * 1000

    # 2. Manual loop convolution
    t0 = time.perf_counter()
    h, w = img.shape
    res_manual = np.zeros_like(img)
    for y in range(1, h - 1):
        for x in range(1, w - 1):
            patch = img[y - 1 : y + 2, x - 1 : x + 2]
            res_manual[y, x] = np.clip(np.sum(patch * kernel), 0, 255)
    t_manual = (time.perf_counter() - t0) * 1000

    print(f"filter2D speed: {t_cv:.4f} ms | Manual loop speed: {t_manual:.2f} ms")

In [ ]:
# Solution 2: Bilateral filter sigmaColor sweep
# Explanation: As `sigmaColor` increases, pixels with larger color differences are blended
# together, smoothing out color boundaries. A low `sigmaColor` (e.g. 10) preserves sharp color
# differences, whereas a high value (e.g. 150) starts turning bilateral filtering into simple
# Gaussian blur, washing out fine color boundaries.


In [ ]:
# Solution 3: Combined Median + Bilateral filter noise cleaning pipeline
def clean_noisy_image(image: np.ndarray) -> np.ndarray:
    """Denoise image using sequential median and bilateral filtering."""
    # First: Median filter removes extreme outliers (salt-and-pepper noise)
    median_filtered = cv2.medianBlur(image, 5)
    # Second: Bilateral filter removes white noise while preserving edges
    return cv2.bilateralFilter(median_filtered, 9, 75, 75)


benchmark_convolution()
# Test Solution 3
img_test = load_real_image("images/standard", "peppers.jpg")
noise_test = np.random.randint(0, 2, img_test.shape, dtype=np.uint8) * 255
noisy_test = cv2.bitwise_or(img_test, noise_test)
cleaned_test = clean_noisy_image(noisy_test)
show_grid([("Noisy", noisy_test), ("Cleaned with Solution 3", cleaned_test)])

## Summary

You can match a filter to a noise model and explain the trade-off between smoothing, edge preservation, and runtime.

- **Best Practices:** Start with the smallest useful kernel, compare against an unchanged original, and measure the effect on the downstream task—not only appearance.
- **Common Pitfalls:** Using a large blur to hide every problem, applying a filter repeatedly, and forgetting that bilateral filtering can be expensive.